# Evaluator v3 — final models, MPC Protocol 1, seeds + paired bootstrap, ACR saliency check

Scores every trained model it finds in the attached notebook outputs:
`<job>_best.pth` + `<job>_meta.json` from **raftnet-fullrun** (RAFT-Net, ForMa, ablations) and
`cflnet_e9_best.pth` from **e9-cflnet-baseline**. Nothing to configure: models are discovered automatically.

**Protocol (MPC Protocol 1):** images and masks resized to 512×512, per-image F1 / IoU at 0.5 averaged over
tampered images; CASIA v1, Columbia, IMD2020 (+ CASIA v2 validation clean and under JPEG QF90/70/50/30 and
the OSN chain). Checkpoints were chosen on CASIA v2 validation only — test sets are used here for the first time.

**Statistics:** mean ± std over seeds; paired bootstrap 95% CIs over test images for RAFT-Net vs ForMa (per
seed) and for every ablation vs the full model of the same seed; noise floor from identical-config runs.

**ACR check:** Grad-CAM saliency energy, fraction of zero pixels, raw and normalised Saliency-TV for every
RAFT-Net variant — detects the suspected collapse of the ACR penalty to an empty saliency map.

**Incremental:** results are saved per model. Evaluate accounts 1–4 now; later, attach this notebook's
previous output + the new models' outputs, and only the new models are evaluated.

**Inputs:** CASIA v2 (divg07), CASIA v1 images (sophatvathana) + CASIA 1.0 groundtruth (+ optionally the
renamed "CASIA 1.0 dataset" to score all ~920 images), Columbia, IMD2020, and the *outputs* of the training
notebooks (Add Input → Notebooks). **Settings:** GPU, Internet on.


## 1. Setup + FCSS patch

In [ ]:
import subprocess, sys, os

REPO_DIR = '/kaggle/working/ForMa_repo'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                     'https://github.com/multimediaFor/ForMa.git', REPO_DIR], check=True)

MODELS_DIR = os.path.join(REPO_DIR, 'models')
print('Repo cloned at:', REPO_DIR)
print('Models dir:', MODELS_DIR, '| exists:', os.path.isdir(MODELS_DIR))


# -- Fix: ForMa's own csm_triton.py has an unguarded @triton.jit decorator --------------
try:
    import triton
    print("triton already available:", triton.__version__)
except ImportError:
    print("triton not installed -- attempting pip install...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'triton'],
                    capture_output=True, text=True)
    try:
        import triton
        print("triton installed successfully:", triton.__version__)
    except ImportError:
        print("triton still unavailable -- injecting a no-op stub so csm_triton.py can import "
              "without crashing. Safe: WITH_TRITON correctly evaluates False in that file, so "
              "the actual computation already uses the verified pure-PyTorch scan regardless "
              "of this stub.")
        import types
        _triton_stub = types.ModuleType('triton')
        _triton_stub.jit = lambda fn=None, **kwargs: (fn if fn is not None else (lambda f: f))
        _tl_stub = types.ModuleType('triton.language')
        _tl_stub.constexpr = object
        _triton_stub.language = _tl_stub
        sys.modules['triton'] = _triton_stub
        sys.modules['triton.language'] = _tl_stub


In [ ]:
import importlib, subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

for pkg, imp in [('timm', 'timm'), ('fvcore', 'fvcore'), ('einops', 'einops'), ('opencv-python-headless', 'cv2')]:
    try:
        importlib.import_module(imp)
    except ImportError:
        pip_install(pkg)

# mamba-ssm / causal-conv1d: ForMa's own code (csms6s.py) gracefully falls back to a
# pure-PyTorch selective-scan if no CUDA kernel is importable -- a failed install here
# costs speed, not correctness.
try:
    import selective_scan_cuda  # noqa: F401
    print("mamba-ssm's selective_scan_cuda already importable.")
except ImportError:
    print("Installing mamba-ssm + causal-conv1d (compiles CUDA kernels, can take a few minutes)...")
    try:
        pip_install('causal-conv1d>=1.2.0', 'mamba-ssm', '--no-build-isolation')
        import selective_scan_cuda  # noqa: F401
        print("mamba-ssm installed and selective_scan_cuda is importable.")
    except Exception as e:
        print("mamba-ssm install/import did not succeed:", e)
        print("Not fatal for THIS model -- falls back to a pure-PyTorch scan (slower, correct).")


In [ ]:
vmamba_path = os.path.join(MODELS_DIR, 'vmamba_pixelshuf_modals.py')
with open(vmamba_path, 'r') as f:
    src = f.read()

patch1_old = '''with open('../config_pixelshuf_modals.yaml', 'r') as f:
    args = yaml.safe_load(f)
device = get_device(args["cuda_idx"])'''
patch1_new = '''args = {"cuda_idx": 0}  # only args["cuda_idx"] is referenced elsewhere in this file
device = get_device(0)'''

patch2_old = '''        with torch.cuda.device(args["cuda_idx"]):
            y = self.forward_core(x)'''
patch2_new = '''        if torch.cuda.is_available():
            with torch.cuda.device(args["cuda_idx"]):
                y = self.forward_core(x)
        else:
            y = self.forward_core(x)'''

# Patch 3: SS2D's default forward_type "v2" hardcodes selective_scan_backend="oflex" --
# VMamba's own separate custom CUDA kernel, never installed here. mamba-ssm's kernel
# (backend="mamba") DID import successfully -- swapping the string is a pure bugfix.
patch3_old = '''            v2=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="oflex"),#core
            v3=partial(self.forward_corev2, force_fp32=False, selective_scan_backend="oflex"),'''
patch3_new = '''            v2=partial(self.forward_corev2, force_fp32=(not self.disable_force32), selective_scan_backend="mamba"),#core -- patched: oflex kernel unavailable, mamba backend verified working
            v3=partial(self.forward_corev2, force_fp32=False, selective_scan_backend="mamba"),  # patched, same reason'''

# Patch 5: a leftover debug print fires on every forward pass, flooding the log.
patch5_old = '''        x = self.patch_embed(x)#[4, 128, 128, 96]
        print(x.shape,"=================")'''
patch5_new = '''        x = self.patch_embed(x)#[4, 128, 128, 96]'''

n_patches = 0
if patch1_old in src:
    src = src.replace(patch1_old, patch1_new); n_patches += 1
if patch2_old in src:
    src = src.replace(patch2_old, patch2_new); n_patches += 1
if patch3_old in src:
    src = src.replace(patch3_old, patch3_new); n_patches += 1
if patch5_old in src:
    src = src.replace(patch5_old, patch5_new); n_patches += 1

with open(vmamba_path, 'w') as f:
    f.write(src)

print(f"Applied {n_patches}/4 patches to {vmamba_path}")
assert n_patches == 4, "Expected patch text not found -- ForMa's repo may have changed upstream."

# Patch 4: dispatch-fallback safety net in csms6s.py, defense in depth beyond patch 3.
csms6s_path = os.path.join(MODELS_DIR, 'csms6s.py')
with open(csms6s_path, 'r') as f:
    csms6s_src = f.read()

patch4_old = '''    WITH_CUDA = (WITH_SELECTIVESCAN_OFLEX or WITH_SELECTIVESCAN_CORE or WITH_SELECTIVESCAN_MAMBA)
    fn = selective_scan_torch if backend == "torch" or (not WITH_CUDA) else SelectiveScanCuda.apply
    return fn(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex, backend)'''
patch4_new = '''    WITH_CUDA = (WITH_SELECTIVESCAN_OFLEX or WITH_SELECTIVESCAN_CORE or WITH_SELECTIVESCAN_MAMBA)
    _avail = {"oflex": WITH_SELECTIVESCAN_OFLEX, "core": WITH_SELECTIVESCAN_CORE, "mamba": WITH_SELECTIVESCAN_MAMBA}
    if backend == "torch" or not WITH_CUDA:
        backend = "torch"
    elif backend not in _avail or not _avail.get(backend, False):
        backend = next((b for b in ("mamba", "core", "oflex") if _avail[b]), "torch")
    fn = selective_scan_torch if backend == "torch" else SelectiveScanCuda.apply
    return fn(u, delta, A, B, C, D, delta_bias, delta_softplus, oflex, backend)'''

n_patches_4 = 0
if patch4_old in csms6s_src:
    csms6s_src = csms6s_src.replace(patch4_old, patch4_new)
    n_patches_4 = 1

with open(csms6s_path, 'w') as f:
    f.write(csms6s_src)

print(f"Applied {n_patches_4}/1 dispatch-fallback patch to {csms6s_path}")
assert n_patches_4 == 1, "Expected patch text not found in csms6s.py -- ForMa's repo may have changed upstream."


In [ ]:
# ── FCSS: patch VMamba's selective scan so a per-position forensic bias can steer the step size ──
# Must run BEFORE vmamba_pixelshuf_modals is imported (each SS2D binds its scan function at construction).
# A model that never sets `forensic_bias` (the ForMa baseline) runs exactly as before.
_vm = os.path.join(MODELS_DIR, 'vmamba_pixelshuf_modals.py')
_src = open(_vm).read()
_anchor = "            xs = xs.view(B, -1, L)\n            dts = dts.contiguous().view(B, -1, L)\n"
_inject = _anchor + (
    "            _fb = getattr(self, 'forensic_bias', None)  # RAFT-Net FCSS\n"
    "            if _fb is not None:\n"
    "                if tuple(_fb.shape[-2:]) != (H, W):\n"
    "                    _fb = F.interpolate(_fb, size=(H, W), mode='bilinear', align_corners=False)\n"
    "                _fbs = cross_scan_fn(_fb.to(x.dtype).contiguous(), in_channel_first=True, out_channel_first=True,\n"
    "                                     scans=_scan_mode, force_torch=scan_force_torch)\n"
    "                dts = dts + _fbs.contiguous().view(B, -1, L).to(dts.dtype)\n")
if 'RAFT-Net FCSS' in _src:
    print('FCSS patch already present')
else:
    assert _src.count(_anchor) == 1, 'FCSS anchor not found exactly once -- ForMa code changed?'
    open(_vm, 'w').write(_src.replace(_anchor, _inject))
    print('FCSS patch applied to', _vm)


## 2. Configuration

In [ ]:
import os, sys, glob, json, math, time, random, gc, shutil, warnings
warnings.filterwarnings('ignore')
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

cv2.setNumThreads(0)
device = torch.device('cuda')
assert torch.cuda.is_available(), 'turn the GPU on'
INPUT, BATCH, NUM_WORKERS = 512, 8, 4
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
SEARCH_ROOT = '/kaggle/input'
SAVE_DIR = '/kaggle/working/eval_v3'
RES_DIR = os.path.join(SAVE_DIR, 'results')
os.makedirs(RES_DIR, exist_ok=True)
ONLY_JOBS = None            # e.g. ['raftnet_s1221', 'forma_s1221']; None = every model found
RUN_DEGRADATION = True
RUN_SALIENCY, SALIENCY_N = True, 300
BOOTSTRAP_B = 2000
CASIA2_ROOT = '/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2'
print('GPU:', torch.cuda.get_device_name(0))


## 3. Model code

In [ ]:
MODEL_SRC = r'''
# raftnet.py -- RAFT-Net and the official ForMa baseline, built on ForMa's own classes.
import torch
import torch.nn as nn
import torch.nn.functional as F
from vmamba_pixelshuf_modals import (Backbone_VSSM, SegFormerHead, ModalMixer, ModalitiesExtractor, ConvModule)

IMNET_MEAN = (0.485, 0.456, 0.406)
IMNET_STD = (0.229, 0.224, 0.225)


class Denorm(nn.Module):
    """Model input is ImageNet-normalised (for the backbone); forensic extractors need raw [0,1] pixels
    (Noiseprint++ was trained on [0,1] RGB). Applied identically in ForMa and RAFT-Net."""
    def __init__(self):
        super().__init__()
        self.register_buffer('mean', torch.tensor(IMNET_MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(IMNET_STD).view(1, 3, 1, 1))

    def forward(self, x):
        return x * self.std + self.mean


def conv_bn_relu(cin, cout, k=3, s=1):
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, k // 2, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


# ── Official ForMa baseline ───────────────────────────────────────────────────────────────────
class ForMaOfficial(nn.Module):
    """Forensic_Vmamba as released: VMamba-T backbone; Noiseprint++ (frozen) + Bayar + SRM on the image
    shrunk to 1/4, mixed to 3 channels, fed to the decoder only."""
    def __init__(self, np_weights):
        super().__init__()
        self.denorm = Denorm()
        self.backbone = Backbone_VSSM()
        self.decode = SegFormerHead()
        self.modalmixer = ModalMixer()
        self.modal_extractor = ModalitiesExtractor(['noiseprint', 'bayar', 'srm'], np_weights)

    def train(self, mode=True):
        super().train(mode)
        self.modal_extractor.noiseprint.eval()     # ForMa's model.train() would silently re-enable BN updates
        return self

    def forward(self, x, dct=None, return_feat=False):
        n, _, h, w = x.shape
        y = self.backbone(x)
        xm = F.interpolate(self.denorm(x), size=(h // 4, w // 4), mode='bilinear', align_corners=False)
        xm = self.modalmixer(self.modal_extractor.multi_output(xm))
        return {'mask': self.decode(y, xm)}


class ForMaWide(ForMaOfficial):
    """Official ForMa with ONE change: the width w of its forensic pathway. ForMa's mixer compresses 96 channels of
    noise features to 3 (then linearly expands them to 16 before fusion with 384 RGB channels). Here the mixer outputs
    w channels and w channels enter the fusion layer; backbone, extractors, mixer blocks and decoder are unchanged."""
    def __init__(self, np_weights, width):
        super().__init__(np_weights)
        self.modalmixer = ModalMixer(out_channels=width)
        self.decode.conv_modals = nn.Conv2d(width, width, kernel_size=1)
        self.decode.linear_fuse = ConvModule(c1=96 * 4 + width, c2=96, k=1)


# ── RAFT-Net components ───────────────────────────────────────────────────────────────────────
class ForensicStem(nn.Module):
    """Forensic evidence at every backbone scale (1/4, 1/8, 1/16, 1/32).
    fullres=True : SRM + Bayar residuals computed on the FULL-resolution image, downsampled by learned
                   stride-2 convolutions (ForMa computes them on a bilinearly shrunk image, which blurs away
                   the fine residuals these filters exist to capture).
    Noiseprint++ : frozen camera-fingerprint net, at 1/4 as in ForMa (full resolution is ~16x too slow on a T4).
    dct=True     : native-grid block-DCT map (computed in the data loader on the original-resolution image,
                   JPEG grid intact) injected at 1/8."""
    def __init__(self, np_weights, fullres=True, dct=True, widths=(64, 128, 192, 256)):
        super().__init__()
        self.fullres, self.use_dct, self.widths = fullres, dct, widths
        self.hp = ModalitiesExtractor(['bayar', 'srm'])
        self.npp = ModalitiesExtractor(['noiseprint'], np_weights)
        self.hp_down = (nn.Sequential(conv_bn_relu(6, 32, 3, 2), conv_bn_relu(32, 48, 3, 2)) if fullres
                        else conv_bn_relu(6, 48, 3, 1))
        self.np_conv = conv_bn_relu(3, 16, 3, 1)
        self.f4 = conv_bn_relu(48 + 16, widths[0], 3, 1)
        self.d8 = conv_bn_relu(widths[0], widths[1], 3, 2)
        if dct:
            self.dct_enc = nn.Sequential(conv_bn_relu(64, widths[1], 1, 1), conv_bn_relu(widths[1], widths[1], 3, 1))
        self.d16 = conv_bn_relu(widths[1], widths[2], 3, 2)
        self.d32 = conv_bn_relu(widths[2], widths[3], 3, 2)

    def forward(self, x01, dct=None):
        h, w = x01.shape[-2:]
        q = F.interpolate(x01, size=(h // 4, w // 4), mode='bilinear', align_corners=False)
        hp = torch.cat(self.hp.multi_output(x01 if self.fullres else q), dim=1)
        f4 = self.f4(torch.cat([self.hp_down(hp), self.np_conv(self.npp.multi_output(q)[0])], dim=1))
        f8 = self.d8(f4)
        if self.use_dct:
            if dct is None:
                raise ValueError('RAFT-Net with dct=True needs the native-grid DCT map as input')
            if dct.shape[-2:] != f8.shape[-2:]:
                dct = F.interpolate(dct, size=f8.shape[-2:], mode='nearest')
            f8 = f8 + self.dct_enc(dct.to(f8.dtype))
        f16 = self.d16(f8)
        return [f4, f8, f16, self.d32(f16)]


class FCSS(nn.Module):
    """Forensic-Conditioned Selective Scan. For every VMamba block, a ZERO-initialised 1x1 conv maps the stage's
    forensic features to a bias on the SSM step size (dt) at every position, bounded by SCALE*tanh(.). The patched
    SS2D.forward_corev2 adds it (cross-scanned into all 4 scan directions) before the selective scan. Zero init =>
    the network starts exactly as the ImageNet-pretrained VMamba; training learns how far forensic evidence
    should steer the scan."""
    SCALE = 2.0

    def __init__(self, backbone, in_widths):
        super().__init__()
        convs, self._ops = [], []
        for s, layer in enumerate(backbone.layers):
            for blk in layer.blocks:
                d_inner = blk.op.dt_projs_weight.shape[1]
                c = nn.Conv2d(in_widths[s], d_inner, 1)
                nn.init.zeros_(c.weight)
                nn.init.zeros_(c.bias)
                convs.append(c)
                self._ops.append((s, blk.op))
        self.convs = nn.ModuleList(convs)

    def set(self, feats):
        for (s, op), conv in zip(self._ops, self.convs):
            op.forensic_bias = self.SCALE * torch.tanh(conv(feats[s]))

    def clear(self):
        for _, op in self._ops:
            op.forensic_bias = None


class WideHead(SegFormerHead):
    """ForMa's pixel-shuffle decoder, with the forensic pathway widened (ForMa: 3 -> 16 channels next to 384 RGB
    channels). Same computation as SegFormerHead.forward; optionally also returns the forensic features (for ACR)."""
    def __init__(self, modal_in=64, modal_out=32, embedding_dim=96):
        super().__init__(embedding_dim=embedding_dim)
        self.conv_modals = nn.Conv2d(modal_in, modal_out, kernel_size=1)
        self.linear_fuse = ConvModule(c1=embedding_dim * 4 + modal_out, c2=embedding_dim, k=1)

    def forward(self, inputs, x_modals, return_feat=False):
        c1, c2, c3, c4 = inputs
        m = self.conv_modals(x_modals)
        n = c4.shape[0]
        _c4 = F.pixel_shuffle(self.linear_c4(c4).permute(0, 2, 1).reshape(n, -1, c4.shape[2], c4.shape[3]), 8)
        _c3 = F.pixel_shuffle(self.linear_c3(c3).permute(0, 2, 1).reshape(n, -1, c3.shape[2], c3.shape[3]), 4)
        _c2 = F.pixel_shuffle(self.linear_c2(c2).permute(0, 2, 1).reshape(n, -1, c2.shape[2], c2.shape[3]), 2)
        _c1 = self.linear_c1(c1).permute(0, 2, 1).reshape(n, -1, c1.shape[2], c1.shape[3])
        out = self.linear_pred(self.dropout(self.linear_fuse(torch.cat([_c4, _c3, _c2, _c1, m], dim=1))))
        return (out, m) if return_feat else out


class BoundaryModule(nn.Module):
    """BASNet-style boundary-motivated module (from the RAFT-Net prototype, unchanged)."""
    def __init__(self, c_low1=96, c_low2=192, c_high=768, mid_channels=64):
        super().__init__()
        self.conv_f1 = nn.Sequential(nn.Conv2d(c_low1, mid_channels, 1), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True))
        self.conv_f2 = nn.Sequential(nn.Conv2d(c_low2, mid_channels, 1), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True))
        self.gate = nn.Sequential(nn.Conv2d(mid_channels * 2, mid_channels, 1), nn.ReLU(inplace=True),
                                  nn.Conv2d(mid_channels, mid_channels, 1))
        self.attn = nn.Sequential(nn.Conv2d(mid_channels, mid_channels, 1), nn.Sigmoid())
        self.conv_high = nn.Sequential(nn.Conv2d(c_high, mid_channels, 1), nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True))

    def forward(self, f1, f2, f_high):
        hw = f1.shape[-2:]
        F1 = self.conv_f1(f1)
        F2 = self.conv_f2(F.interpolate(f2, size=hw, mode='bilinear', align_corners=True))
        FL = self.attn(self.gate(torch.cat([F1, F2], dim=1)))
        FH = self.conv_high(F.interpolate(f_high, size=hw, mode='bilinear', align_corners=True))
        return FL * FH


class EdgeEnhance(nn.Module):
    """BASNet-style boundary-enhanced module (from the RAFT-Net prototype, unchanged)."""
    def __init__(self, channels=64):
        super().__init__()
        self.pool1 = nn.AdaptiveAvgPool2d(1)
        self.pool2 = nn.AdaptiveAvgPool2d(4)
        self.atrous = nn.Conv2d(channels, channels, kernel_size=3, padding=2, dilation=2)
        self.fuse = nn.Conv2d(channels * 2, channels, kernel_size=1)
        self.swish = nn.SiLU(inplace=True)

    def forward(self, x):
        h, w = x.shape[-2:]
        p1 = F.interpolate(self.pool1(x), size=(h, w), mode='bilinear', align_corners=True)
        p2 = F.interpolate(self.pool2(x), size=(h, w), mode='bilinear', align_corners=True)
        return x + self.swish(self.atrous(x)) * torch.sigmoid(self.fuse(torch.cat([p1, p2], dim=1)))


class BoundaryBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.bmm = BoundaryModule()
        self.bem = EdgeEnhance(64)
        self.head = nn.Sequential(conv_bn_relu(64, 32, 3, 1), nn.Conv2d(32, 1, 1))

    def forward(self, c1, c2, c4, out_hw):
        b = self.head(self.bem(self.bmm(c1, c2, c4)))
        return F.interpolate(b, size=out_hw, mode='bilinear', align_corners=True)


# ── RAFT-Net ──────────────────────────────────────────────────────────────────────────────────
class RAFTNet(nn.Module):
    def __init__(self, np_weights, fcss=True, dct=True, fullres=True, boundary=True):
        super().__init__()
        self.cfg = dict(fcss=fcss, dct=dct, fullres=fullres, boundary=boundary)
        self.denorm = Denorm()
        self.backbone = Backbone_VSSM()
        self.stem = ForensicStem(np_weights, fullres=fullres, dct=dct)
        self.fcss = FCSS(self.backbone, self.stem.widths) if fcss else None
        self.decode = WideHead(modal_in=self.stem.widths[0], modal_out=32)
        self.boundary = BoundaryBranch() if boundary else None

    def train(self, mode=True):
        super().train(mode)
        self.stem.npp.noiseprint.eval()
        return self

    def forward(self, x, dct=None, return_feat=False):
        h, w = x.shape[-2:]
        feats = self.stem(self.denorm(x), dct)
        if self.fcss is not None:
            self.fcss.set(feats)
        try:
            y = self.backbone(x)
        finally:
            if self.fcss is not None:
                self.fcss.clear()
        out = {}
        if return_feat:
            out['mask'], out['feat'] = self.decode(y, feats[0], return_feat=True)
        else:
            out['mask'] = self.decode(y, feats[0])
        if self.boundary is not None:
            out['boundary'] = self.boundary(y[0], y[1], y[3], (h, w))
        return out


VARIANTS = {
    # name                  builder kwargs (None = official ForMa)                                    needs DCT input
    'forma':               (None,                                                                     False),
    'raftnet':             (dict(fcss=True,  dct=False, fullres=True,  boundary=True),                False),  # FINAL
    'raftnet_no_fcss':     (dict(fcss=False, dct=False, fullres=True,  boundary=True),                False),
    'raftnet_no_boundary': (dict(fcss=True,  dct=False, fullres=True,  boundary=False),               False),
    'raftnet_no_fullres':  (dict(fcss=True,  dct=False, fullres=False, boundary=True),                False),
    'raftnet_lite_nofullres': (dict(fcss=False, dct=False, fullres=False, boundary=True),             False),  # final candidate
    'forma_w12':           (dict(width=12),                                                            False),  # bottleneck study
    'forma_w48':           (dict(width=48),                                                            False),  # bottleneck study
}


def build_model(variant, np_weights):
    kw, _ = VARIANTS[variant]
    if kw is None:
        return ForMaOfficial(np_weights)
    if 'width' in kw:
        return ForMaWide(np_weights, kw['width'])
    return RAFTNet(np_weights, **kw)


def load_imagenet_backbone(model, sd, prefix='backbone.'):
    """ForMa train.py logic: copy checkpoint tensors into model[prefix + name] when shapes match."""
    msd = model.state_dict()
    n = 0
    for k, v in sd.items():
        mk = prefix + k
        if mk in msd and tuple(msd[mk].shape) == tuple(v.shape):
            msd[mk] = v
            n += 1
    model.load_state_dict(msd)
    bb = [k for k in msd if k.startswith(prefix)]
    return n, len(sd), len(bb)
'''


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm


In [ ]:
# --- model/srm.py, verbatim (source: Kishor-Bhaumik/CFLNet, adapted from
#     github.com/selimsef/dfdc_deepfake_challenge; kernels from Zhou et al. CVPR 2018) ---

def setup_srm_weights(input_channels: int = 3, output_channel: int = 1) -> torch.Tensor:
    srm_kernel = torch.from_numpy(
        np.array([
            [  # srm 1/2 horiz
                [0., 0., 0., 0., 0.],
                [0., 0., 0., 0., 0.],
                [0., 1., -2., 1., 0.],
                [0., 0., 0., 0., 0.],
                [0., 0., 0., 0., 0.],
            ],
            [  # srm 1/4
                [0., 0., 0., 0., 0.],
                [0., -1., 2., -1., 0.],
                [0., 2., -4., 2., 0.],
                [0., -1., 2., -1., 0.],
                [0., 0., 0., 0., 0.],
            ],
            [  # srm 1/12
                [-1., 2., -2., 2., -1.],
                [2., -6., 8., -6., 2.],
                [-2., 8., -12., 8., -2.],
                [2., -6., 8., -6., 2.],
                [-1., 2., -2., 2., -1.],
            ]
        ])).float()
    srm_kernel[0] /= 2
    srm_kernel[1] /= 4
    srm_kernel[2] /= 12
    return srm_kernel.view(3, 1, 5, 5).repeat(output_channel, input_channels, 1, 1)


def setup_srm_layer(input_channels: int = 3, output_channel=None) -> torch.nn.Module:
    if output_channel is None:
        weights = setup_srm_weights(input_channels)
        conv = torch.nn.Conv2d(input_channels, out_channels=3, kernel_size=5, stride=1, padding=2, bias=False)
    else:
        weights = setup_srm_weights(input_channels, output_channel)
        conv = torch.nn.Conv2d(input_channels, out_channels=output_channel, kernel_size=5, stride=1, padding=2, bias=False)
    with torch.no_grad():
        conv.weight = torch.nn.Parameter(weights, requires_grad=False)
    return conv


In [ ]:
# --- model/aspp.py, verbatim ---

class _ASPPModule(nn.Module):
    def __init__(self, inplanes, planes, kernel_size, padding, dilation, BatchNorm):
        super(_ASPPModule, self).__init__()
        self.atrous_conv = nn.Conv2d(inplanes, planes, kernel_size=kernel_size,
                                      stride=1, padding=padding, dilation=dilation, bias=False)
        self.bn = BatchNorm(planes)
        self.relu = nn.ReLU()
        self._init_weight()

    def forward(self, x):
        x = self.atrous_conv(x)
        x = self.bn(x)
        return self.relu(x)

    def _init_weight(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                torch.nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()


class ASPP(nn.Module):
    def __init__(self, inplanes=2048, outplanes=512, output_stride=16, BatchNorm=nn.BatchNorm2d):
        super(ASPP, self).__init__()
        if output_stride == 16:
            dilations = [1, 6, 12, 18]
        elif output_stride == 8:
            dilations = [1, 12, 24, 36]
        else:
            raise NotImplementedError

        self.aspp1 = _ASPPModule(inplanes, outplanes, 1, padding=0, dilation=dilations[0], BatchNorm=BatchNorm)
        self.aspp2 = _ASPPModule(inplanes, outplanes, 3, padding=dilations[1], dilation=dilations[1], BatchNorm=BatchNorm)
        self.aspp3 = _ASPPModule(inplanes, outplanes, 3, padding=dilations[2], dilation=dilations[2], BatchNorm=BatchNorm)
        self.aspp4 = _ASPPModule(inplanes, outplanes, 3, padding=dilations[3], dilation=dilations[3], BatchNorm=BatchNorm)

        self.global_avg_pool = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)),
                                              nn.Conv2d(inplanes, outplanes, 1, stride=1, bias=False),
                                              BatchNorm(outplanes),
                                              nn.ReLU())
        self.conv1 = nn.Conv2d(outplanes * 5, outplanes, 1, bias=False)
        self.bn1 = BatchNorm(outplanes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self._init_weight()

    def forward(self, x):
        x1 = self.aspp1(x)
        x2 = self.aspp2(x)
        x3 = self.aspp3(x)
        x4 = self.aspp4(x)
        x5 = self.global_avg_pool(x)
        x5 = F.interpolate(x5, size=x4.size()[2:], mode='bilinear', align_corners=True)
        x = torch.cat((x1, x2, x3, x4, x5), dim=1)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        return self.dropout(x)

    def _init_weight(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                torch.nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()


def build_aspp(inplanes=2048, outplanes=512, output_stride=16, BatchNorm=nn.BatchNorm2d):
    return ASPP(inplanes, outplanes, output_stride, BatchNorm)


In [ ]:
# --- model/model.py, same architecture as the official repo. The only change is that
#     BOTH encoders read `pretrained` from CFG instead of the RGB one being hardcoded True. ---

class CFLNet(nn.Module):
    def __init__(self, cfg, inplanes):
        super(CFLNet, self).__init__()
        self.cfg = cfg
        self.encoder = timm.create_model(
            cfg['model_params']['encoder'],
            pretrained=cfg['model_params']['pretrained_rgb'],   # official repo hardcodes True here
            features_only=True, out_indices=[4],
        )
        self.conv_srm = setup_srm_layer()
        self.encoder_srm = timm.create_model(
            cfg['model_params']['encoder'],
            pretrained=cfg['model_params']['pretrained_srm'],   # official repo hardcodes False here
            features_only=True, out_indices=[4],
        )
        if cfg['global_params']['with_srm']:
            self.aspp = build_aspp(inplanes=inplanes * 2, outplanes=cfg['model_params']['aspp_outplane'])
        else:
            self.aspp = build_aspp(inplanes=inplanes, outplanes=cfg['model_params']['aspp_outplane'])

        self.decoder = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1, stride=1, bias=False),
            nn.BatchNorm2d(512),
            nn.Conv2d(512, cfg['model_params']['num_class'], kernel_size=1, stride=1, bias=True),
        )
        self.projection = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=1),
            nn.BatchNorm2d(512),
            nn.Conv2d(512, 256, kernel_size=1),
        )

    def forward(self, inp):
        x = self.encoder(inp)[0]
        if self.cfg['global_params']['with_srm']:
            x_srm = self.conv_srm(inp)
            x_srm = self.encoder_srm(x_srm)[0]
            x = torch.cat([x, x_srm], dim=1)
        x = self.aspp(x)
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=True)
        out = self.decoder(x)
        proj = self.projection(x)
        return out, proj


In [ ]:
with open(os.path.join(MODELS_DIR, 'raftnet.py'), 'w') as f:
    f.write(MODEL_SRC)
if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)
import importlib, raftnet
importlib.reload(raftnet)
from raftnet import build_model
NP_WEIGHTS = os.path.join(REPO_DIR, 'weights', 'np++.pth')

CFL_EVAL_CFG = {'global_params': {'with_srm': True, 'with_con': True},
                'model_params': {'encoder': 'resnet50', 'aspp_outplane': 512, 'num_class': 2,
                                 'pretrained_rgb': False, 'pretrained_srm': False}}   # weights come from the checkpoint
with torch.no_grad():
    _p = timm.create_model('resnet50', pretrained=False, features_only=True, out_indices=[4])
    CFL_IN_PLANES = _p(torch.randn(2, 3, 128, 128))[0].shape[1]
    del _p
print('model code ready | CFL-Net encoder channels:', CFL_IN_PLANES)


## 4. Metric engine

In [ ]:
# ── Unified metric engine ─────────────────────────────────────────────────────────
# Pure numpy, histogram-based: every pixel's tampered-probability is binned into 1000 bins, so
# counts at ANY threshold k/1000 are exact cumulative sums -- no pixel subsampling, bounded
# memory regardless of image size, and pixel ROC-AUC / AP computed on ALL pixels.
#
# Threshold semantics: a pixel is predicted tampered at threshold t=k/1000 iff prob >= t,
# which is exactly floor(prob*1000) >= k. At k=500 this is the standard fixed 0.5 threshold.
#
# Reported per dataset (tampered images WITH a ground-truth mask only, the Protocol-1
# convention used by MPC / PSCC-Net / TruFor tables):
#   F1/IoU/P/R per-image @0.5, averaged over images   <- HEADLINE (matches MPC Table II/III)
#   F1/IoU pooled over all pixels @0.5
#   F1/IoU pooled at the dataset-best threshold       (optimistic: threshold chosen on test data)
#   F1 per-image at each image's own best threshold   (oracle upper bound, clearly labelled)
#   pixel ROC-AUC and AP (all pixels of tampered images)
# plus image-level detection AUC (max-pooled prob), using authentic images as negatives.
import numpy as np
from sklearn.metrics import roc_auc_score

N_BINS = 1000
K_05 = 500  # bin index for threshold 0.5


def _safe_div(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    out = np.zeros(np.broadcast(a, b).shape, dtype=np.float64)
    np.divide(a, b, out=out, where=b > 0)
    return out


class MetricAccumulator:
    def __init__(self, name):
        self.name = name
        self.hist_pos = np.zeros(N_BINS, dtype=np.int64)   # GT-tampered pixels, all tampered images
        self.hist_neg = np.zeros(N_BINS, dtype=np.int64)   # GT-authentic pixels of tampered images
        self.rows = []            # per tampered image with a usable mask
        self.img_labels = []      # image-level: 1 tampered, 0 authentic
        self.img_scores = []      # image-level score: max prob
        self.n_empty_gt = 0       # "tampered" images whose mask has zero tampered pixels
        self.n_no_mask = 0        # tampered images with no mask file (image-level only)

    def add(self, prob, gt, label, key=None):
        """prob: HxW float in [0,1] at the SAME resolution as gt.
        gt: HxW {0,1} array, or None (authentic image, or tampered image without a mask).
        label: 1 tampered / 0 authentic (image level)."""
        prob = np.asarray(prob, dtype=np.float32)
        self.img_labels.append(int(label))
        self.img_scores.append(float(prob.max()))
        if gt is None:
            if label == 1:
                self.n_no_mask += 1
            return
        g = np.asarray(gt).astype(bool)
        assert g.shape == prob.shape, f"shape mismatch prob {prob.shape} vs gt {g.shape}"
        n_pos = int(g.sum())
        if n_pos == 0:
            self.n_empty_gt += 1
            return
        idx = np.minimum((prob * N_BINS).astype(np.int64), N_BINS - 1)
        hp = np.bincount(idx[g], minlength=N_BINS)
        hn = np.bincount(idx[~g], minlength=N_BINS)
        self.hist_pos += hp
        self.hist_neg += hn
        tp = np.cumsum(hp[::-1])[::-1]          # tp[k]: GT-pos pixels with prob >= k/1000
        fp = np.cumsum(hn[::-1])[::-1]
        fn = n_pos - tp
        f1_all = _safe_div(2 * tp, 2 * tp + fp + fn)
        TP, FP, FN = int(tp[K_05]), int(fp[K_05]), int(fn[K_05])
        self.rows.append({
            'key': key,
            'f1': float(_safe_div(2 * TP, 2 * TP + FP + FN)),
            'iou': float(_safe_div(TP, TP + FP + FN)),
            'precision': float(_safe_div(TP, TP + FP)),
            'recall': float(_safe_div(TP, TP + FN)),
            'f1_best_own_thr': float(f1_all.max()),
            'gt_frac': n_pos / g.size,
            'f1_all_tampered': 2 * n_pos / (2 * n_pos + (g.size - n_pos)),   # trivial baseline: every pixel predicted tampered
            'pred_frac': (TP + FP) / g.size,
            'tp': TP, 'fp': FP, 'fn': FN,
        })

    def summary(self):
        s = {'dataset': self.name, 'n_images_total': len(self.img_labels),
             'n_tampered_scored': len(self.rows), 'n_empty_gt_skipped': self.n_empty_gt,
             'n_tampered_without_mask': self.n_no_mask,
             'n_authentic': int(sum(1 for l in self.img_labels if l == 0))}
        if self.rows:
            for k in ['f1', 'iou', 'precision', 'recall', 'f1_best_own_thr', 'gt_frac', 'pred_frac', 'f1_all_tampered']:
                s[f'img_mean_{k}'] = float(np.mean([r[k] for r in self.rows]))
            TP = sum(r['tp'] for r in self.rows)
            FP = sum(r['fp'] for r in self.rows)
            FN = sum(r['fn'] for r in self.rows)
            s['pooled_f1@0.5'] = float(_safe_div(2 * TP, 2 * TP + FP + FN))
            s['pooled_iou@0.5'] = float(_safe_div(TP, TP + FP + FN))
            tp = np.cumsum(self.hist_pos[::-1])[::-1]
            fp = np.cumsum(self.hist_neg[::-1])[::-1]
            fn = self.hist_pos.sum() - tp
            f1 = _safe_div(2 * tp, 2 * tp + fp + fn)
            kb = int(np.argmax(f1))
            s['pooled_best_thr'] = kb / N_BINS
            s['pooled_f1@best'] = float(f1[kb])
            s['pooled_iou@best'] = float(_safe_div(tp[kb], tp[kb] + fp[kb] + fn[kb]))
            n_pos, n_neg = self.hist_pos.sum(), self.hist_neg.sum()
            if n_pos > 0 and n_neg > 0:
                # ROC-AUC = P(score_pos > score_neg) + 0.5 * P(tie within a bin)
                neg_below = np.concatenate([[0], np.cumsum(self.hist_neg)[:-1]])
                s['pixel_roc_auc'] = float((self.hist_pos * (neg_below + 0.5 * self.hist_neg)).sum()
                                           / (float(n_pos) * float(n_neg)))
                # AP = sum over thresholds (high -> low) of precision * delta-recall
                prec = _safe_div(tp, tp + fp)
                rec = tp / n_pos
                rec_next = np.concatenate([rec[1:], [0.0]])
                s['pixel_ap'] = float(((rec - rec_next) * prec).sum())
        labels = np.array(self.img_labels)
        if len(set(labels.tolist())) == 2:
            s['image_auc_maxpool'] = float(roc_auc_score(labels, np.array(self.img_scores)))
        return s


## 5. Dataset helpers

In [ ]:
# ── Dataset loaders: every entry is (image_path, mask_path_or_None, label, mask_kind) ──────
# mask_kind tells load_gt() how to turn the mask file into {0,1}:
#   'binary'   -> grayscale > 127 (CASIA v1/v2, IMD2020)
#   'columbia' -> colour-coded region map: tampered = pixels where the COLUMBIA_TAMPERED
#                 colour channel dominates (NOT grayscale>127, which keeps only the thin
#                 bright-green boundary band -- the bug behind the old Columbia numbers)
import os, glob, json, random
import numpy as np
import cv2

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp')
COLUMBIA_TAMPERED = 'green'   # verify against the README printed in the Columbia cell


def load_gt(mask_path, mask_kind, target_hw):
    """Returns a {0,1} uint8 mask at target_hw=(H, W) of the ORIGINAL image, or None."""
    if mask_path is None:
        return None
    if mask_kind == 'columbia':
        bgr = cv2.imread(mask_path, cv2.IMREAD_COLOR)
        if bgr is None:
            return None
        g = bgr[..., 1].astype(np.int16)
        r = bgr[..., 2].astype(np.int16)
        m = (g > r) if COLUMBIA_TAMPERED == 'green' else (r > g)
        m = m.astype(np.uint8)
    else:
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            return None
        m = (m > 127).astype(np.uint8)
    if m.shape[:2] != tuple(target_hw):
        m = cv2.resize(m, (target_hw[1], target_hw[0]), interpolation=cv2.INTER_NEAREST)
    return m


# ── CASIA v2 in-distribution split ─────────────────────────────────────────────────────
# 'validated' = the split used by every RAFT-Net stage, MPC and LoMa (unreadable files such as
#               Thumbs.db dropped BEFORE shuffling -> 12472 samples -> 11225/1247).
# 'cfl'       = CFL-Net's split (nothing dropped -> 12473 samples incl. Thumbs.db -> 11226/1247).
# The two share only ~41% of their validation images, so CFL-Net must be scored on its OWN
# split (the other one contains ~59% of CFL-Net's training images).
def casia2_file_list(au_dir, tp_dir, gt_dir):
    samples = []
    for f in sorted(os.listdir(au_dir)):
        p = os.path.join(au_dir, f)
        if os.path.isfile(p) and not f.startswith('.'):
            samples.append((p, None))
    gt_by_stem = {}
    for f in os.listdir(gt_dir):
        if f.startswith('.'):
            continue
        stem = os.path.splitext(f)[0]
        if stem.endswith('_gt'):
            stem = stem[:-3]
        gt_by_stem[stem] = os.path.join(gt_dir, f)
    for f in sorted(os.listdir(tp_dir)):
        p = os.path.join(tp_dir, f)
        if not (os.path.isfile(p) and not f.startswith('.')):
            continue
        m = gt_by_stem.get(os.path.splitext(f)[0])
        if m is not None:
            samples.append((p, m))
    return samples


def casia2_val_split(au_dir, tp_dir, gt_dir, variant, seed=1221, val_fraction=0.1,
                     readable_check=None):
    samples = casia2_file_list(au_dir, tp_dir, gt_dir)
    if variant == 'validated':
        assert readable_check is not None
        samples = [s for s in samples if readable_check(s)]
    elif variant != 'cfl':
        raise ValueError(variant)
    random.Random(seed).shuffle(samples)
    n_val = int(len(samples) * val_fraction)
    val, train = samples[:n_val], samples[n_val:]
    # CFL's list can contain an unreadable file in val; it is skipped at load time exactly as
    # CFL-Net's own evaluation did (reported as skipped).
    as_entries = lambda lst: [(img, m, 0 if m is None else 1, 'binary') for img, m in lst]
    return as_entries(val), as_entries(train)


# ── Columbia ────────────────────────────────────────────────────────────────────────────
def columbia_entries(root):
    splc_img = [d for d in glob.glob(os.path.join(root, '**', '4cam_splc'), recursive=True)
                if any(f.lower().endswith(('.tif', '.tiff')) for f in os.listdir(d))]
    auth_img = [d for d in glob.glob(os.path.join(root, '**', '4cam_auth'), recursive=True)
                if any(f.lower().endswith(('.tif', '.tiff')) for f in os.listdir(d))]
    edge_dirs = [d for d in glob.glob(os.path.join(root, '**', 'edgemask'), recursive=True)
                 if '4cam_splc' in d]
    assert splc_img and auth_img and edge_dirs, (splc_img, auth_img, edge_dirs)
    edge = {}
    for d in edge_dirs:
        for f in os.listdir(d):
            stem = os.path.splitext(f)[0]
            if stem.endswith('_edgemask'):          # '_edgemask_3' variants are ignored
                edge[stem[:-len('_edgemask')]] = os.path.join(d, f)
    entries, unmatched = [], []
    for f in sorted(os.listdir(splc_img[0])):
        if not f.lower().endswith(('.tif', '.tiff')):
            continue
        stem = os.path.splitext(f)[0]
        m = edge.get(stem)
        if m is None:
            unmatched.append(f)
        entries.append((os.path.join(splc_img[0], f), m, 1, 'columbia'))
    for f in sorted(os.listdir(auth_img[0])):
        if f.lower().endswith(('.tif', '.tiff')):
            entries.append((os.path.join(auth_img[0], f), None, 0, 'binary'))
    return entries, unmatched


# ── IMD2020 ─────────────────────────────────────────────────────────────────────────────
def imd2020_entries(root):
    auth_dir, img_dir, mask_dir = (os.path.join(root, d) for d in ('Authentic', 'img', 'mask'))
    masks = {}
    for f in os.listdir(mask_dir):
        stem = os.path.splitext(f)[0]
        if stem.lower().endswith('_mask'):
            masks[stem[:-5]] = os.path.join(mask_dir, f)
    entries, unmatched = [], []
    for f in sorted(os.listdir(img_dir)):
        if not f.lower().endswith(IMG_EXTS):
            continue
        m = masks.get(os.path.splitext(f)[0])
        if m is None:
            unmatched.append(f)
        entries.append((os.path.join(img_dir, f), m, 1, 'binary'))
    for f in sorted(os.listdir(auth_dir)):
        if f.lower().endswith(IMG_EXTS):
            entries.append((os.path.join(auth_dir, f), None, 0, 'binary'))
    return entries, unmatched


# ── CASIA v1 ────────────────────────────────────────────────────────────────────────────
# The attached image mirror ships NO masks. Masks come from a separate groundtruth dataset
# (e.g. the community "CASIA 1.0 groundtruth" set: Sp_..._gt.png). Discovery is by NAME,
# not by folder layout: any image file under /kaggle/input whose stem (minus an optional
# '_gt' suffix) equals a CASIA1 tampered-image stem is taken as that image's mask.
def casia1_entries(casia1_root, search_root='/kaggle/input'):
    sp_dir = os.path.join(casia1_root, 'Sp')
    au_dir = os.path.join(casia1_root, 'Au')
    tamp = {os.path.splitext(f)[0]: os.path.join(sp_dir, f)
            for f in sorted(os.listdir(sp_dir)) if f.lower().endswith(IMG_EXTS)}
    masks = {}
    for r, dirs, files in os.walk(search_root):
        if os.path.abspath(r).startswith(os.path.abspath(sp_dir)):
            continue
        for f in files:
            if not f.lower().endswith(IMG_EXTS):
                continue
            stem = os.path.splitext(f)[0]
            base = stem[:-3] if stem.endswith('_gt') else stem
            if base in tamp and (stem.endswith('_gt') or 'truth' in r.lower() or 'mask' in r.lower()
                                 or 'gt' in os.path.basename(r).lower()):
                masks[base] = os.path.join(r, f)
    entries = [(p, masks.get(s), 1, 'binary') for s, p in tamp.items()]
    entries += [(os.path.join(au_dir, f), None, 0, 'binary')
                for f in sorted(os.listdir(au_dir)) if f.lower().endswith(IMG_EXTS)]
    return entries, sum(1 for s in tamp if s not in masks)


## 6. Discover datasets and models

In [ ]:
# ── datasets ──────────────────────────────────────────────────────────────────────────────
def find_attached(name, base=SEARCH_ROOT):
    for r, _, fs in os.walk(base):
        if name in fs:
            return os.path.join(r, name)
    return None

def find_dirs(pattern):
    return sorted([p for p in glob.glob(f'{SEARCH_ROOT}/**/{pattern}', recursive=True) if os.path.isdir(p)], key=len)

DATASETS = {}
_sj = find_attached('casia2_split_seed1221.json')
if _sj:
    VAL = [tuple(x) for x in json.load(open(_sj))['val']]
else:
    from concurrent.futures import ThreadPoolExecutor
    _au, _tp, _gt = f'{CASIA2_ROOT}/Au', f'{CASIA2_ROOT}/Tp', f'{CASIA2_ROOT}/CASIA 2 Groundtruth'
    _all = casia2_file_list(_au, _tp, _gt)
    def _readable(s):
        return cv2.imread(s[0]) is not None and (s[1] is None or cv2.imread(s[1]) is not None)
    with ThreadPoolExecutor(8) as ex:
        _bad = {s for s, ok in zip(_all, ex.map(_readable, _all)) if not ok}
    _v, _ = casia2_val_split(_au, _tp, _gt, 'validated', readable_check=lambda s: s not in _bad)
    VAL = [(a, b) for a, b, _, _ in _v]
assert len(VAL) == 1247, len(VAL)
DATASETS['casia2'] = [(a, b, 0 if b is None else 1, 'binary') for a, b in VAL]

def _norm(p):
    return p.lower().replace(' ', '').replace('_', '').replace('-', '')

def casia1_entries_v3(root=SEARCH_ROOT):
    """Tampered = every 'Sp_*' image outside ground-truth folders (works for the original mirror and the
    renamed CASIA 1.0 release); mask = same stem (+ optional '_gt') in any ground-truth folder."""
    tamp, masks, auth = {}, {}, {}
    for r, _, fs in os.walk(root):
        rn = _norm(r)
        gt_dir = ('truth' in rn) or ('mask' in rn)
        for f in fs:
            if not f.lower().endswith(IMG_EXTS):
                continue
            stem, p = os.path.splitext(f)[0], os.path.join(r, f)
            if stem.endswith('_gt') or (gt_dir and stem.startswith('Sp_')):
                masks.setdefault(stem[:-3] if stem.endswith('_gt') else stem, p)
            elif stem.startswith('Sp_') and not gt_dir:
                tamp.setdefault(stem, p)
            elif stem.startswith('Au_') and 'casia1' in rn and 'casia2' not in rn:
                auth.setdefault(stem, p)
    matched = [s for s in tamp if s in masks]
    keep = matched if len(matched) >= 900 else list(tamp)      # renamed release present -> drop old-name duplicates
    ents = [(tamp[s], masks.get(s), 1, 'binary') for s in sorted(keep)]
    ents += [(p, None, 0, 'binary') for _, p in sorted(auth.items())]
    return ents, len(matched), len(tamp)

DATASETS['casia1'], _nm, _nt = casia1_entries_v3()
print(f"CASIA v1: tampered {sum(e[2] for e in DATASETS['casia1'])} (with mask {_nm}; MPC used 920) | "
      f"authentic {sum(1 - e[2] for e in DATASETS['casia1'])}")
col_root = next((os.path.dirname(d) for d in find_dirs('4cam_splc') if os.path.isdir(os.path.join(os.path.dirname(d), '4cam_auth'))), None)
if col_root:
    DATASETS['columbia'], _ = columbia_entries(col_root)
imd_root = next((d for d in find_dirs('IMD2020') if all(os.path.isdir(os.path.join(d, s)) for s in ('Authentic', 'img', 'mask'))), None)
if imd_root:
    DATASETS['imd2020'], _ = imd2020_entries(imd_root)
for k, v in DATASETS.items():
    print(f"{k:<9}: {sum(e[2] == 1 and e[1] is not None for e in v)} tampered with mask | {len(v)} images total")

# ── models: every <job>_meta.json (raftnet-fullrun) and cflnet_e9_meta.json (e9) in the attached outputs ──
def discover_models(root=SEARCH_ROOT):
    found = {}
    for r, _, fs in os.walk(root):
        for f in fs:
            if not f.endswith('_meta.json'):
                continue
            try:
                meta = json.load(open(os.path.join(r, f)))
            except Exception:
                continue
            if 'variant' in meta and 'job' in meta:
                job = meta['job']
                fam = 'forma' if meta['variant'].startswith('forma') else 'raftnet'
                best, hist = os.path.join(r, f'{job}_best.pth'), os.path.join(r, f'{job}_history.json')
                st = os.path.join(r, f'{job}_status.json')
                state = json.load(open(st)).get('state', '?') if os.path.exists(st) else '?'
            elif meta.get('model') == 'CFL-Net':
                job, fam = 'cflnet_e9', 'cfl'
                best, hist = os.path.join(r, 'cflnet_e9_best.pth'), os.path.join(os.path.dirname(r), 'cflnet_e9_history.json')
                n_ep = len(json.load(open(hist))) if os.path.exists(hist) else 0
                state = 'done' if n_ep >= meta.get('epochs', 40) else f'{n_ep} epochs'
            else:
                continue
            if not os.path.exists(best):
                continue
            score = (meta.get('best_val') or {}).get('val_f1', -1.0)
            if job not in found or score > found[job]['score']:
                found[job] = dict(job=job, family=fam, variant=meta.get('variant', 'cflnet'), seed=meta.get('seed', 1221),
                                  acr=bool(meta.get('acr', False)), best=best, history=hist if os.path.exists(hist) else None,
                                  score=score, best_epoch=meta.get('best_epoch'), state=state)
    return found

MODELS = discover_models()
if ONLY_JOBS:
    MODELS = {k: v for k, v in MODELS.items() if k in ONLY_JOBS}
for k in sorted(MODELS):
    m = MODELS[k]
    print(f"{k:<28} {m['variant']:<20} seed {m['seed']} acr {int(m['acr'])} | best val F1 {m['score']:.4f} "
          f"(epoch {m['best_epoch']}) | training {m['state']}" + ("" if m['state'] == 'done' else "   <-- NOT FINISHED"))

# previous evaluation results attached? reuse them (incremental evaluation)
for r, _, fs in os.walk(SEARCH_ROOT):
    if os.path.basename(r) == 'results' and 'eval_v3' in r:
        for f in fs:
            if not os.path.exists(os.path.join(RES_DIR, f)):
                shutil.copy(os.path.join(r, f), os.path.join(RES_DIR, f))
DONE = {f[:-5] for f in os.listdir(RES_DIR) if f.endswith('.json')}
print('already evaluated (reused):', sorted(DONE) or 'none')


## 7. Inference

In [ ]:
# ── inference at 512 (MPC protocol) ─────────────────────────────────────────────────────────
def jpeg_degrade(qf):
    def _fn(img):
        ok, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, qf])
        return cv2.imdecode(enc, cv2.IMREAD_COLOR) if ok else img
    return _fn

def osn_chain_degrade(scale=0.7, qf=65):
    def _fn(img):
        h, w = img.shape[:2]
        small = cv2.resize(img, (max(1, int(w * scale)), max(1, int(h * scale))), interpolation=cv2.INTER_AREA)
        ok, enc = cv2.imencode('.jpg', small, [cv2.IMWRITE_JPEG_QUALITY, qf])
        if ok:
            small = cv2.imdecode(enc, cv2.IMREAD_COLOR)
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    return _fn

DEGRADATIONS = {'jpeg_qf90': jpeg_degrade(90), 'jpeg_qf70': jpeg_degrade(70), 'jpeg_qf50': jpeg_degrade(50),
                'jpeg_qf30': jpeg_degrade(30), 'osn_chain': osn_chain_degrade()}

class EvalSet(Dataset):
    def __init__(self, entries, degrade=None):
        self.entries, self.degrade = entries, degrade
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, i):
        ip, mp, label, kind = self.entries[i]
        bgr = cv2.imread(ip, cv2.IMREAD_COLOR)
        if bgr is None:
            return None
        gt = load_gt(mp, kind, bgr.shape[:2])
        if self.degrade is not None:                     # degradation at native resolution, before the resize
            bgr = self.degrade(bgr)
        x = cv2.resize(bgr, (INPUT, INPUT))[..., ::-1].astype(np.float32) / 255.0
        x = np.ascontiguousarray(((x - MEAN) / STD).transpose(2, 0, 1))
        g = None if gt is None else cv2.resize(gt, (INPUT, INPUT), interpolation=cv2.INTER_NEAREST)
        return torch.from_numpy(x), g, int(label), ip

def _collate(b):
    return [x for x in b if x is not None]

def build(e):
    m = build_model(e['variant'], NP_WEIGHTS) if e['family'] in ('raftnet', 'forma') else CFLNet(CFL_EVAL_CFG, CFL_IN_PLANES)
    sd = torch.load(e['best'], map_location='cpu', weights_only=False)
    sd = sd.get('model', sd) if isinstance(sd, dict) and 'model' in sd and isinstance(sd['model'], dict) else sd
    sd = {(k[7:] if k.startswith('module.') else k): v for k, v in sd.items()}
    missing, unexpected = m.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, f"{e['job']}: missing {len(missing)} unexpected {len(unexpected)}"
    return m.to(device).eval()

def logits_of(m, x):
    out = m(x)
    return out['mask'] if isinstance(out, dict) else out[0]

@torch.no_grad()
def run_eval(m, entries, name, degrade=None):
    import gc as _gc
    _gc.collect()
    acc = MetricAccumulator(name)
    for batch in DataLoader(EvalSet(entries, degrade), batch_size=BATCH, num_workers=NUM_WORKERS, collate_fn=_collate):
        if not batch:
            continue
        x = torch.stack([b[0] for b in batch]).to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            lg = logits_of(m, x)
        d = F.interpolate((lg[:, 1:2] - lg[:, 0:1]).float(), (INPUT, INPUT), mode='bilinear', align_corners=True)
        for p, (_, g, lab, key) in zip(torch.sigmoid(d)[:, 0].cpu().numpy(), batch):
            acc.add(p, g, lab, key=key)
    return acc.summary(), {r['key']: (r['f1'], r['iou']) for r in acc.rows}


## 8. ACR saliency check

In [ ]:
# ── ACR check: Grad-CAM saliency statistics on CASIA v2 validation (RAFT-Net variants) ───────
def _tv(S, prob, thr=0.5):
    m_hat = (prob > thr).float()
    dh, dw = (S[:, :, 1:] - S[:, :, :-1]).abs(), (S[:, :, :, 1:] - S[:, :, :, :-1]).abs()
    mh = ((m_hat[:, :, 1:] + m_hat[:, :, :-1]) > 0).float()
    mw = ((m_hat[:, :, :, 1:] + m_hat[:, :, :, :-1]) > 0).float()
    return ((dh * mh).sum((1, 2, 3)) + (dw * mw).sum((1, 2, 3))) / m_hat.sum((1, 2, 3)).clamp(min=1.0)

def saliency_stats(m, entries, n=SALIENCY_N):
    ents = [e for e in entries if e[2] == 1 and e[1] is not None][:n]
    out = {k: [] for k in ('energy', 'peak', 'frac_zero', 'tv_raw', 'tv_norm', 'region_frac')}
    m.eval()
    for batch in DataLoader(EvalSet(ents), batch_size=4, num_workers=NUM_WORKERS, collate_fn=_collate):
        x = torch.stack([b[0] for b in batch]).to(device)
        with torch.enable_grad(), torch.amp.autocast('cuda', enabled=False):
            o = m(x.float(), None, return_feat=True)
            feat, lg = o['feat'], o['mask'].float()
            g = torch.autograd.grad(lg[:, 1].sum(), feat)[0]
            S = F.relu((g.mean(dim=(2, 3), keepdim=True) * feat).sum(dim=1, keepdim=True)).detach()
            prob = F.softmax(lg, dim=1)[:, 1:2].detach()
        peak = S.flatten(1).max(1)[0]
        Sn = S / (peak.view(-1, 1, 1, 1) + 1e-12)
        out['energy'] += S.flatten(1).mean(1).tolist()
        out['peak'] += peak.tolist()
        out['frac_zero'] += (S.flatten(1) <= 1e-12).float().mean(1).tolist()
        out['tv_raw'] += _tv(S, prob).tolist()
        out['tv_norm'] += _tv(Sn, prob).tolist()
        out['region_frac'] += (prob > 0.5).float().flatten(1).mean(1).tolist()
    res = {k: float(np.mean(v)) for k, v in out.items()}
    res['degenerate_frac'] = float(np.mean(np.array(out['peak']) < 1e-8))
    res['n'] = len(out['peak'])
    return res


## 9. Evaluate

In [ ]:
# ── evaluate every model not evaluated yet (results saved per model -> safe to interrupt) ─────
order = sorted(MODELS, key=lambda j: (MODELS[j]['family'] != 'raftnet', j))
for job in order:
    if job in DONE:
        continue
    e = MODELS[job]
    t0 = time.time()
    print(f"\n### {job} ({e['variant']}, seed {e['seed']}, acr {int(e['acr'])}, training {e['state']})")
    m = build(e)
    res = dict(job=job, family=e['family'], variant=e['variant'], seed=e['seed'], acr=e['acr'], state=e['state'],
               best_epoch=e['best_epoch'], val_f1_train=e['score'], history=e['history'],
               params_m=sum(p.numel() for p in m.parameters()) / 1e6, rows=[])
    per = {}
    for ds in ('casia2', 'casia1', 'columbia', 'imd2020'):
        if ds not in DATASETS:
            continue
        s, pi = run_eval(m, DATASETS[ds], ds)
        res['rows'].append({**s, 'dataset': ds, 'level': 'clean'})
        per[f'{ds}__clean'] = pi
        print(f"  {ds:<9} clean     F1 {s.get('img_mean_f1', float('nan')) * 100:5.1f} | IoU {s.get('img_mean_iou', float('nan')) * 100:5.1f} | "
              f"pixel AUC {s.get('pixel_roc_auc', float('nan')):.3f} | n {s['n_tampered_scored']}")
    if RUN_DEGRADATION:
        for lvl, fn in DEGRADATIONS.items():
            s, pi = run_eval(m, DATASETS['casia2'], f'casia2_{lvl}', fn)
            res['rows'].append({**s, 'dataset': 'casia2', 'level': lvl})
            per[f'casia2__{lvl}'] = pi
            print(f"  casia2    {lvl:<9} F1 {s['img_mean_f1'] * 100:5.1f} | IoU {s['img_mean_iou'] * 100:5.1f}")
    if RUN_SALIENCY and e['family'] == 'raftnet':
        res['saliency'] = saliency_stats(m, DATASETS['casia2'])
        sa = res['saliency']
        print(f"  saliency: energy {sa['energy']:.3e} | zero pixels {sa['frac_zero'] * 100:.1f}% | "
              f"degenerate maps {sa['degenerate_frac'] * 100:.1f}% | TV raw {sa['tv_raw']:.4f} | TV normalised {sa['tv_norm']:.4f}")
    json.dump(res, open(os.path.join(RES_DIR, f'{job}.json'), 'w'), indent=1, default=float)
    arrs = {}
    for k, pi in per.items():
        keys = sorted(pi)
        arrs[f'{k}__keys'] = np.array(keys)
        arrs[f'{k}__f1'] = np.array([pi[q][0] for q in keys], dtype=np.float32)
        arrs[f'{k}__iou'] = np.array([pi[q][1] for q in keys], dtype=np.float32)
    np.savez_compressed(os.path.join(RES_DIR, f'{job}_perimage.npz'), **arrs)
    print(f"  saved | {(time.time() - t0) / 60:.1f} min")
    del m
    gc.collect(); torch.cuda.empty_cache()


## 10. Tables

In [ ]:
# ── analysis (numpy / pandas only) ──────────────────────────────────────────────────────────
import pandas as pd

CITED = {   # MPC, Tables II-III (Protocol 1), per-image F1 / IoU %, as (CASIA v1, Columbia, IMD2020)
    'FCN': ((44.1, 22.3, 21.0), (36.7, 17.7, 15.8)), 'DeepLabv3': ((42.9, 44.2, 21.6), (36.1, 35.3, 15.9)),
    'ManTra-Net': ((18.7, 45.2, 16.4), (11.1, 30.1, 9.8)), 'SPAN': ((14.3, 50.3, 14.5), (11.2, 39.0, 10.0)),
    'PSCC-Net': ((33.5, 50.3, 19.7), (23.2, 36.0, 12.0)), 'MVSS-Net++': ((51.3, 66.0, 27.0), (39.7, 57.3, 20.0)),
    'CAT-Net': ((23.7, 20.6, 25.7), (16.5, 14.0, 18.3)), 'TruFor': ((53.2, 82.9, 35.9), (48.1, 78.1, 29.7)),
    'PIM': ((56.6, 68.0, 41.9), (51.2, 60.4, 34.0)), 'MPC': ((44.8, 67.6, 48.5), (41.2, 61.7, 40.1))}
CROSS = ['casia1', 'columbia', 'imd2020']
LEVELS = ['clean', 'jpeg_qf90', 'jpeg_qf70', 'jpeg_qf50', 'jpeg_qf30', 'osn_chain']

def load_results(res_dir):
    R, P = {}, {}
    for f in sorted(os.listdir(res_dir)):
        if f.endswith('.json'):
            job = f[:-5]
            R[job] = json.load(open(os.path.join(res_dir, f)))
            z = np.load(os.path.join(res_dir, f'{job}_perimage.npz'))
            P[job] = {k[:-6]: {_canon(k[:-6], q): f for q, f in zip(z[k], z[k[:-6] + '__f1'])}
                      for k in z.files if k.endswith('__keys')}
    return R, P

def group_of(r):
    if r['family'] == 'cfl':
        return 'CFL-Net'
    if r['family'] == 'forma':
        return 'ForMa' if r['variant'] == 'forma' else f"ForMa (w={r['variant'].split('_w')[-1]})"
    if r['variant'] == 'raftnet' and r['acr']:
        return 'RAFT-Net'
    if r['variant'] == 'raftnet_no_fcss' and not r['acr']:
        return 'RAFT-Net-lite'
    if r['variant'] == 'raftnet_lite_nofullres':
        return 'RAFT-Net-lite w/o full-res'
    tag = {'raftnet': 'w/o ACR', 'raftnet_no_fcss': 'w/o FCSS', 'raftnet_no_boundary': 'w/o boundary head',
           'raftnet_no_fullres': 'w/o full-res stem'}[r['variant']]
    return f'RAFT-Net {tag}'

def metric(r, ds, lvl='clean', key='img_mean_f1'):
    for row in r['rows']:
        if row['dataset'] == ds and row['level'] == lvl:
            return row.get(key, float('nan')) * 100
    return float('nan')

def weighted_cross(r, key='img_mean_f1'):
    num = den = 0.0
    for row in r['rows']:
        if row['dataset'] in CROSS and row['level'] == 'clean' and row.get('n_tampered_scored'):
            num += row[key] * row['n_tampered_scored']; den += row['n_tampered_scored']
    return 100 * num / den if den else float('nan')

def fmt(vals):
    v = [x for x in vals if x == x]
    if not v:
        return '-'
    return f"{np.mean(v):.1f}" if len(v) == 1 else f"{np.mean(v):.1f} ± {np.std(v, ddof=1):.1f}"

def group_table(R, cols, getter):
    groups = {}
    for job, r in R.items():
        groups.setdefault(group_of(r), []).append(r)
    order = (['RAFT-Net-lite w/o full-res', 'RAFT-Net-lite', 'RAFT-Net', 'ForMa'] + sorted((g for g in groups if g.startswith('ForMa (w=')), key=lambda g: int(g[9:-1]))
             + ['CFL-Net'] + sorted(g for g in groups if g.startswith('RAFT-Net ')))
    rows = []
    for g in order:
        if g in groups:
            rows.append([g, len(groups[g])] + [fmt([getter(r, c) for r in groups[g]]) for c in cols])
    return pd.DataFrame(rows, columns=['model', 'runs'] + cols)

def _canon(ds_level, key):
    # CASIA v1 exists in two attached mirrors (original + renamed release); which copy the discovery picks depends
    # on Kaggle's mount order, so the full path differs between evaluator runs. Pair CASIA v1 images by file stem.
    key = str(key)
    return os.path.splitext(os.path.basename(key))[0] if ds_level.startswith('casia1') else key

PAIRING_WARNINGS = []

def paired_boot(a, b, B=2000, seed=0):
    keys = sorted(set(a) & set(b))
    if not keys:
        return (float('nan'),) * 3 + (0,)
    x = np.array([a[k] - b[k] for k in keys], dtype=np.float64) * 100
    idx = np.random.default_rng(seed).integers(0, len(x), size=(B, len(x)))
    means = x[idx].mean(axis=1)
    return float(x.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)), len(x)

def pooled(P, job, dsets, lvl='clean'):
    d = {}
    for ds in dsets:
        for k, v in P[job].get(f'{ds}__{lvl}', {}).items():
            d[f'{ds}|{k}'] = v
    return d

def paired_table(R, P, pairs, B=2000):
    """pairs: list of (label, job_a, job_b). Difference = a - b in F1 points, 95% bootstrap CI over images."""
    tests = [('CASIA v2 clean', ['casia2'], 'clean'), ('CASIA v2 QF50', ['casia2'], 'jpeg_qf50'),
             ('CASIA v2 OSN', ['casia2'], 'osn_chain'), ('CASIA v1', ['casia1'], 'clean'),
             ('Columbia', ['columbia'], 'clean'), ('IMD2020', ['imd2020'], 'clean'),
             ('cross-dataset (pooled)', CROSS, 'clean')]
    rows = []
    for label, a, b in pairs:
        if a not in P or b not in P:
            continue
        row = [label]
        for _, dsets, lvl in tests:
            da, db = pooled(P, a, dsets, lvl), pooled(P, b, dsets, lvl)
            m, lo, hi, n = paired_boot(da, db, B)
            full = min(len(da), len(db))
            if full and n < 0.98 * full:
                PAIRING_WARNINGS.append(f'{label} | {dsets}: only {n} of {full} images paired')
            row.append('-' if n == 0 else f"{m:+.1f} [{lo:+.1f}, {hi:+.1f}]" + (' *' if (lo > 0 or hi < 0) else ''))
        rows.append(row)
    return pd.DataFrame(rows, columns=['comparison'] + [t[0] for t in tests])

def training_curves(R, out_png):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for grp, color in (('RAFT-Net', 'tab:red'), ('ForMa', 'tab:blue')):
        curves = []
        for r in R.values():
            if group_of(r) == grp and r.get('history') and os.path.exists(r['history']):
                curves.append([h['val_f1'] * 100 for h in json.load(open(r['history']))])
        if curves:
            n = min(len(c) for c in curves)
            a = np.array([c[:n] for c in curves])
            ep = np.arange(1, n + 1)
            ax.plot(ep, a.mean(0), color=color, label=f'{grp} (mean of {len(a)} seeds)')
            if len(a) > 1:
                ax.fill_between(ep, a.min(0), a.max(0), color=color, alpha=0.2)
    ax.set_xlabel('epoch'); ax.set_ylabel('CASIA v2 val per-image F1 (%)'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(out_png, dpi=120)
    return fig

def noise_floor(R):
    """identical-config pairs: same seed, variant 'raftnet', one with ACR one without -> identical until epoch 20"""
    out = []
    for r in R.values():
        if r['variant'] == 'raftnet' and r['acr'] and r.get('history') and os.path.exists(r['history']):
            twin = next((q for q in R.values() if q['variant'] == 'raftnet' and not q['acr'] and q['seed'] == r['seed']
                         and q.get('history') and os.path.exists(q['history'])), None)
            if twin:
                a = [h['val_f1'] for h in json.load(open(r['history']))][:20]
                b = [h['val_f1'] for h in json.load(open(twin['history']))][:20]
                n = min(len(a), len(b))
                d = 100 * (np.array(a[:n]) - np.array(b[:n]))
                out.append((r['seed'], n, float(d.std(ddof=1)), float(np.abs(d).max())))
    return out


In [ ]:
R, P = load_results(RES_DIR)
assert R, 'no evaluation results yet -- attach the training notebooks\' outputs and run the evaluation cell'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
unfinished = [j for j, r in R.items() if r.get('state') != 'done']
if unfinished:
    print('NOTE: evaluated before training finished (re-evaluate later):', unfinished, '\n')

print('=' * 100); print('TABLE 1 -- Protocol 1 cross-dataset, per-image F1 (%), mean ± std over seeds'); print('=' * 100)
t1 = group_table(R, CROSS + ['weighted avg'], lambda r, c: weighted_cross(r) if c == 'weighted avg' else metric(r, c))
print(t1.to_string(index=False))
print('\nSame, IoU (%):')
print(group_table(R, CROSS, lambda r, c: metric(r, c, key='img_mean_iou')).to_string(index=False))
triv = next(iter(R.values()))
print('\ntrivial baseline (every pixel tampered), F1 %:', {c: round(metric(triv, c, key='img_mean_f1_all_tampered'), 1) for c in CROSS})
print('cited (MPC Tables II/III, F1 | IoU):')
for k, (f1, iou) in CITED.items():
    print(f"  {k:<11} F1 {f1[0]:5.1f} {f1[1]:5.1f} {f1[2]:5.1f} | IoU {iou[0]:5.1f} {iou[1]:5.1f} {iou[2]:5.1f}")

print('\n' + '=' * 100); print('TABLE 2 -- robustness on CASIA v2 validation, per-image F1 (%)'); print('=' * 100)
print(group_table(R, LEVELS, lambda r, c: metric(r, 'casia2', c)).to_string(index=False))

seeds = sorted({r['seed'] for r in R.values() if group_of(r) == 'RAFT-Net'})
main_pairs = [(f'RAFT-Net vs ForMa, seed {s}', f'raftnet_s{s}', f'forma_s{s}') for s in seeds]
main_pairs += [(f'RAFT-Net vs CFL-Net, seed {s}', f'raftnet_s{s}', 'cflnet_e9') for s in seeds[:1]]
lite_seeds = sorted({r['seed'] for r in R.values() if group_of(r) == 'RAFT-Net-lite'})
main_pairs += [(f'RAFT-Net-lite vs ForMa, seed {s}', f'raftnet_lite_s{s}', f'forma_s{s}') for s in lite_seeds]
main_pairs += [(f'RAFT-Net-lite vs ForMa w=12, seed {s}', f'raftnet_lite_s{s}', f'forma_w12_s{s}') for s in lite_seeds]
main_pairs += [(f'RAFT-Net-lite vs RAFT-Net (FCSS+ACR), seed {s}', f'raftnet_lite_s{s}', f'raftnet_s{s}') for s in lite_seeds]
lnf_seeds = sorted({r['seed'] for r in R.values() if group_of(r) == 'RAFT-Net-lite w/o full-res'})
main_pairs += [(f'Lite w/o full-res vs ForMa, seed {s}', f'raftnet_lite_nofullres_s{s}', f'forma_s{s}') for s in lnf_seeds]
main_pairs += [(f'Lite w/o full-res vs Lite, seed {s}', f'raftnet_lite_nofullres_s{s}', f'raftnet_lite_s{s}') for s in lnf_seeds]
print('\n' + '=' * 100); print('TABLE 3 -- paired differences, F1 points, 95% bootstrap CI over test images (* = CI excludes 0)'); print('=' * 100)
print(paired_table(R, P, main_pairs, BOOTSTRAP_B).to_string(index=False))

abl = []
for j, r in sorted(R.items()):
    g = group_of(r)
    if g.startswith('RAFT-Net w/o') and f"raftnet_s{r['seed']}" in R:
        abl.append((f"full - ({g[9:]}), seed {r['seed']}", f"raftnet_s{r['seed']}", j))
print('\n' + '=' * 100); print('TABLE 4 -- ablations: full RAFT-Net minus ablated model (same seed), F1 points, 95% CI'); print('=' * 100)
print(paired_table(R, P, abl, BOOTSTRAP_B).to_string(index=False) if abl else 'no ablation results yet')

print('\n' + '=' * 100); print('TABLE 5 -- ACR check: Grad-CAM saliency on CASIA v2 val (RAFT-Net variants)'); print('=' * 100)
sal = [[j, group_of(r), r['saliency']['energy'], r['saliency']['frac_zero'] * 100, r['saliency']['degenerate_frac'] * 100,
        r['saliency']['tv_raw'], r['saliency']['tv_norm']] for j, r in sorted(R.items()) if r.get('saliency')]
print(pd.DataFrame(sal, columns=['job', 'model', 'energy', 'zero px %', 'empty maps %', 'TV raw', 'TV normalised']).to_string(index=False))
print("Reading: if ACR models show far lower energy / many empty maps vs 'w/o ACR', the penalty was satisfied by "
      "switching the saliency off (trivial solution), not by making it coherent.")

wide = sorted((j for j, r in R.items() if r['variant'].startswith('forma_w')), key=lambda j: int(R[j]['variant'].split('_w')[-1]))
if wide:
    print('\n' + '=' * 100); print('TABLE 6 -- forensic bottleneck: widened ForMa minus original ForMa (w=3), same seed, F1 points, 95% CI'); print('=' * 100)
    pairs = [(f"ForMa w={R[j]['variant'].split('_w')[-1]} - ForMa w=3, seed {R[j]['seed']}", j, f"forma_s{R[j]['seed']}") for j in wide]
    print(paired_table(R, P, pairs, BOOTSTRAP_B).to_string(index=False))

print('\npairing check:', 'all paired comparisons matched >= 98% of images' if not PAIRING_WARNINGS
      else '\n  ' + '\n  '.join(PAIRING_WARNINGS))

nf = noise_floor(R)
if nf:
    print('\nnoise floor (identical-config pairs, epochs 1-20): ' +
          '; '.join(f'seed {s}: per-epoch diff std {sd:.2f}, max {mx:.2f} points ({n} epochs)' for s, n, sd, mx in nf))
training_curves(R, os.path.join(SAVE_DIR, 'training_curves.png'))
plt.show()
print('\nparams (M):', {group_of(r): round(r['params_m'], 2) for r in R.values()})
t1.to_csv(os.path.join(SAVE_DIR, 'table1_protocol1.csv'), index=False)
print('\nSaved tables + training_curves.png in', SAVE_DIR)


## What to send back
The five tables printed above (and `training_curves.png`). For a later run with more models: attach this
notebook's previous output as well — finished models are reused and only new ones are evaluated.
